# SLANet → ONNX（Google Colab，x86_64 Linux）

把 PaddleOCR 的表格結構模型 SLANet 轉成 onnxruntime-web 載得進來的 ONNX，並當場驗收三關：
3×4 合成表解出 15 格、4×4 解出 20 格、圖內不含 Loop/If/Scan。

**版本是照 PyPI 上真的有 wheel 的組合挑的**：`paddle2onnx 1.0.8` 只有 cp310/cp311 wheel，
Colab 若是 Python 3.12 就只能用 `2.1.0`（本機 macOS 實測那版會轉出 Loop，屬對照組）。

執行：Runtime → Run all。跑完最後一格會自動下載 `slanet_plus.onnx`。

In [ ]:
import sys, subprocess
print('Python', sys.version)
v = sys.version_info[:2]
# 1.0.8 有 cp310/cp311 wheel；1.0.5 有 cp36-cp311；cp312 只剩 2.1.0
p2o = '1.0.8' if v in [(3,10),(3,11)] else '2.1.0'
paddle = '2.6.2' if v in [(3,10),(3,11),(3,12)] else '3.0.0'
print(f'→ 使用 paddlepaddle=={paddle} + paddle2onnx=={p2o}')
subprocess.run([sys.executable,'-m','pip','install','-q',f'paddlepaddle=={paddle}',f'paddle2onnx=={p2o}',
                'onnx','onnxruntime','pillow','numpy'], check=True)
print('安裝完成')

In [ ]:
# 抓 repo 裡的轉檔腳本（含三關驗收），跑一次
!curl -fsSL https://raw.githubusercontent.com/cormort/pic-calculator/main/bench/experiments/convert_slanet.py -o convert_slanet.py
!python convert_slanet.py --out slanet_plus.onnx --workdir /content/slanet_work

In [ ]:
import os, hashlib, shutil
p = 'slanet_plus.onnx'
if os.path.exists(p):
    print(p, round(os.path.getsize(p)/1024), 'KB', hashlib.sha256(open(p,'rb').read()).hexdigest())
    try:
        from google.colab import files
        files.download(p)   # 自動下載到你的電腦
    except Exception as e:
        print('（不在 Colab 環境，檔案留在', os.getcwd(), '）', e)
else:
    print('沒有產出 ONNX —— 看上一格的 FAIL 訊息')